# Querying the archiver index

A tour of the query API, run against the 216-file `sample/` directory.

This notebook talks to `Store` directly, in process, so **no HSDS server and no
running API are needed** — it ingests the sample into a local `.h5` and reads it
back through the same interface the service uses against HSDS. Every query here
has a one-to-one equivalent over HTTP; see the README's API table.

The sample spans 2026-04-24 to 2026-07-24 and is not representative of the full
archive, so treat the numbers below as illustrative.

## Build the index

Ingest transposes the pile of per-capture metadata files into one column-per-PV
domain. It is written to the system temp directory and reused on later runs;
delete that file to force a rebuild.

In [ ]:
import tempfile
from pathlib import Path

from archiver_search.config import Settings
from archiver_search.ingest import ingest
from archiver_search.models import (
    CameraFilter,
    Clause,
    Group,
    SearchRequest,
    SeriesRequest,
    Sort,
    TimeRange,
)
from archiver_search.store import Store

SAMPLE = Path.cwd().parent / "sample"  # run from examples/; adjust if elsewhere
DOMAIN = Path(tempfile.gettempdir()) / "archiver-sample-index.h5"

# backend="local" opens the domain with h5py. Point `domain` at an HSDS path and
# drop this to run the identical code against a server.
settings = Settings(domain=str(DOMAIN), backend="local")

if not DOMAIN.exists():
    report = ingest(str(SAMPLE), domain=str(DOMAIN), workers=4, settings=settings)
    print(report.summary())
else:
    print(f"reusing {DOMAIN}")

One file in `sample/` is a gzipped tar that was accidentally given a `.hdf`
name. Ingest reports it as skipped and carries on — a 30k-file run must not die
on one bad file.

## What is in there

`meta()` is the cheap orientation call: row count, time bounds, PV count and the
camera list with per-camera row counts.

In [ ]:
store = Store(settings=settings)
meta = store.meta()

print(f"{meta['n_rows']} rows, {meta['pv_count']} PVs")
print(f"{meta['time_from']}  ..  {meta['time_to']}")
print()
for cam in meta["cameras"]:
    print(f"{cam['name']:<22} {cam['area']:<4} {cam['n_rows']:>4} rows   {cam['first']} .. {cam['last']}")

## Finding a PV

`list_pvs` filters the catalogue by area, family, field or a substring. Each
entry carries the min and max recorded at ingest, which is a quick way to pick a
threshold that will actually match something.

In [ ]:
quads = store.list_pvs(area="S04", family="MAG", q="QUAD")
for info in quads:
    print(f"{info.name:<28} {info.min:>10.3f} .. {info.max:>8.3f}")

# Names are forgiving: canonical, sanitised, or an unambiguous bare device name
# with the field left off. All three of these resolve to the same PV.
print()
for token in ("CLA-S04-MAG-QUAD-01:SETI", "cla_s04_mag_quad_01__seti", "CLA-S04-MAG-QUAD-01"):
    print(f"{token:<28} -> {store.resolve_pv(token).name}")

## Query 1 — one PV in a range

The QUAD-01 setpoint between 0 and 15 A. `select` attaches that PV's value to
each returned row.

Watch the two counters in the response: `scanned_rows` is how far the time
window narrowed things before any PV was read, and `pv_reads` is the number of
hyperslabs actually fetched. One predicate on one PV costs exactly one read,
however many rows match.

In [ ]:
def show(result, *, n=5):
    """Print the headline counters and the first few rows of a search result."""
    print(
        f"{result['total']} row(s) · scanned {result['scanned_rows']} · "
        f"{result['pv_reads']} PV read(s) · {result['elapsed_ms']} ms"
    )
    for row in result["rows"][:n]:
        values = "  ".join(
            f"{name.split(':')[-1]}={value:.3f}" if value is not None else f"{name}=nan"
            for name, value in row["values"].items()
        )
        print(f"  {row['id']:>4}  {row['timestamp']}  {row['camera']:<22} {values}")


q1 = store.search(
    SearchRequest(
        filters=Group(
            clauses=[
                Clause(pv="CLA-S04-MAG-QUAD-01:SETI", op="between", value=[0, 15]),
            ]
        ),
        select=["CLA-S04-MAG-QUAD-01:SETI"],
        limit=5,
    )
)
show(q1)

Everything matches: in this sample QUAD-01 never leaves 0 … 11.4 A. Re-run the
cell and `pv_reads` drops to 0 — the column is cached between queries.

## Query 2 — a time window and a camera area

Time and camera are the cheap stages: the window is a binary search over the
sorted timestamp column, the camera filter a mask over a cached integer column.
Neither touches a PV, so `pv_reads` stays at 0.

Bounds given the wrong way round are swapped rather than rejected, and `limit=0`
asks for the count without hydrating any rows.

In [ ]:
april = store.search(
    SearchRequest(
        cameras=CameraFilter(areas=["S07"]),
        time=TimeRange(from_="2026-05-01", to="2026-04-01"),  # deliberately reversed
        limit=0,
    )
)
show(april)

# Nothing in April: the sample's S07 captures are all from June. Drop the window.
everything = store.search(SearchRequest(cameras=CameraFilter(areas=["S07"]), limit=0))
show(everything)

## Query 3 — two conditions, and a nested boolean

Clauses in a `Group` combine with `and` or `or`, and groups nest. `eq` uses an
absolute tolerance (`tol`, default `1e-9`) because these are floats read back
from a control system.

A `pv_pattern` is a glob over several PVs: with `match="any"` the row matches if
any of them satisfies the predicate, with `"all"` if all do.

In [ ]:
both = store.search(
    SearchRequest(
        filters=Group(
            op="and",
            clauses=[
                Clause(pv="CLA-L01-RF-CTRL-01:getPhase", op="gt", value=80),
                Clause(pv="CLA-LRG1-MAG-SOL-01:SETI", op="eq", value=-193.5),
            ],
        ),
        select=["CLA-L01-RF-CTRL-01:getPhase", "CLA-LRG1-MAG-SOL-01:SETI"],
        sort=Sort(by="timestamp", order="desc"),
        limit=5,
    )
)
show(both)

In [ ]:
# QUAD-01 in range AND (any RF phase over 80 OR the L01 power reading missing).
nested = store.search(
    SearchRequest(
        filters=Group(
            op="and",
            clauses=[
                Clause(pv="CLA-S04-MAG-QUAD-01:SETI", op="between", value=[0, 15]),
                Group(
                    op="or",
                    clauses=[
                        Clause(pv_pattern="*:getPhase", match="any", op="gt", value=80),
                        Clause(pv="CLA-L01-RF-CTRL-01:getPower", op="isnan"),
                    ],
                ),
            ],
        ),
        limit=0,
    )
)
show(nested)

## Query 4 — a time series

`series` returns one trace per PV over the matching rows, for plotting. The same
time, camera and filter clauses apply. Series longer than `max_points` are
min/max decimated server-side and flagged with `decimated`.

`{"from": ..., "days": 5}` is the "N days following X" form, so the caller does
no date arithmetic.

In [ ]:
result = store.series(
    SeriesRequest(
        pv_pattern="*:getPhase",
        time=TimeRange(from_="2026-06-24", days=5),
    )
)

print(f"{result['total']} point(s) per trace · {result['elapsed_ms']} ms")
for trace in result["series"]:
    values = [v for v in trace["v"] if v is not None]
    span = f"{min(values):>9.2f} .. {max(values):>8.2f}" if values else "(all nan)"
    print(f"  {trace['pv']:<30} {len(trace['t']):>4} pts  {span}  decimated={trace['decimated']}")

`trace["t"]` holds ISO-8601 timestamps and `trace["v"]` the values, with `None`
where the reading was NaN — ready to hand straight to a plotting library.

## One row in full

`row(id)` returns a single capture with all 182 PVs attached, plus where its
capture file sits relative to the image archive.

In [ ]:
row = store.row(q1["rows"][0]["id"])

print(f"{row['camera']}  {row['timestamp']}  ({row['n_images']} image(s))")
print(f"source   {row['source_file']}")
print(f"capture  {row['capture_path']}")
print(f"mounted  {row['has_image']}")
print(f"\n{len(row['values'])} PV values, e.g.")
for name in sorted(row["values"])[:5]:
    print(f"  {name:<30} {row['values'][name]}")

**Images are referenced, not stored.** The capture files live on the CLARA
storage; the index records only where each sits *relative to the acquisition
host's archive root*, so a reader can mount that archive anywhere. The datasets
inside are named `Capture000001` … `Capture{n_images:06d}`:

```python
import h5py

with h5py.File(f"{image_root}/{row['capture_path']}") as f:
    image = f["Capture000001"][:]   # greyscale
```

`has_image` above is `True` only when this host actually has that file: set
`ARCHIVER_IMAGE_ROOT` (or `Settings(image_root=...)`) to the mount point. It is
`False` here because the sample ships no image data — only the metadata files.

## Exporting

`iter_rows` streams every matching row in chunks, ignoring the page limit. This
is what backs `GET /api/export.csv`.

In [ ]:
request = SearchRequest(
    cameras=CameraFilter(areas=["S01"]),
    select=["CLA-S04-MAG-QUAD-01:SETI"],
    limit=0,
)

total = 0
for block in store.iter_rows(request, chunk=50):
    total += len(block)
    print(f"block of {len(block)}, first is row {block[0]['id']}")
print(f"{total} row(s) streamed")

In [ ]:
store.close()

## The same queries over HTTP

With the service running (`archiver-api`, or `python -m archiver_search`), every
query above is also a request. The structured body mirrors `SearchRequest`
field for field:

```bash
curl -sG localhost:8000/api/search \
     --data-urlencode 'f=CLA-S04-MAG-QUAD-01:SETI|between|0|15'

curl -s localhost:8000/api/search -H 'content-type: application/json' \
     -d '{"filters": {"op": "and", "clauses": [
           {"pv": "CLA-S04-MAG-QUAD-01:SETI", "op": "between", "value": [0, 15]}]},
          "select": ["CLA-S04-MAG-QUAD-01:SETI"], "limit": 5}'
```

Interactive docs are at `/docs`.